In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import subprocess, sys

# Fix fairseq2 for T4 (cu126 environment)
subprocess.check_call([sys.executable, "-m", "pip", "install", 
    "fairseq2==0.7.0", "fairseq2n==0.7.0",
    "--extra-index-url", "https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu126",
    "--force-reinstall", "--no-deps", "-q"])

print("✅ fairseq2 ready")

In [ ]:
!sed -i '194s/_check_torch_version()/#_check_torch_version()/' /usr/local/lib/python3.12/dist-packages/fairseq2n/__init__.py

In [ ]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

models_to_try = [
    "omniASR_CTC_300M_v2",
    "omniASR_LLM_300M_v2", 
    "omniASR_LLM_300M",
    "omniASR_CTC_300M",
]

for name in models_to_try:
    try:
        print(f"Trying: {name}")
        p = ASRInferencePipeline(model_card=name)
        print(f"✅ SUCCESS: {name}")
        break
    except Exception as e:
        print(f"❌ Failed: {e}\n")

In [ ]:
import sys
import subprocess
import importlib
print("=" * 60)
print("KIKUYU YOUTUBE TRANSCRIBER - OMNILINGUAL ASR (FIXED)")
print("=" * 60)

# Step 1: Setup
print("\n🔧 Setting up environment...")
def install_package(package):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        return True
    except:
        return False

install_package("omnilingual-asr")
install_package("yt-dlp")
install_package("pydub")
install_package("gradio")
install_package("tqdm")
install_package("soundfile")

print("\n✅ Packages installed!")

# Imports
import torch
import torchaudio
import json
import os
from pathlib import Path
from datetime import datetime
import tempfile
import numpy as np
import yt_dlp
from pydub import AudioSegment
from tqdm import tqdm
import gradio as gr
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

print("\n✅ Imports OK")

# Hardware
print(f"\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

class KikuyuTranscriber:
    def __init__(self, output_dir="/kaggle/working/kikuyu_transcriptions", model_size="300M", prefer_ctc=True):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.audio_dir = self.output_dir / "audio"
        self.audio_dir.mkdir(exist_ok=True)
        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Model selection logic
        variants = [
            f"omniASR_LLM_{model_size}",  # This is the working one
            f"omniASR_CTC_{model_size}_v2",
            f"omniASR_LLM_{model_size}_v2",
        ]
        if model_size == "300M" and not prefer_ctc:
            variants.insert(0, "omniASR_CTC_300M_v2")  # fallback
        
        self.pipeline = None
        self.model_card = None
        
        print(f"\nTrying to load model ({model_size})...")
        for name in variants:
            try:
                print(f"  → {name}")
                self.pipeline = ASRInferencePipeline(model_card=name)
                self.model_card = name
                print(f"✅ Loaded: {name}")
                break
            except Exception as e:
                print(f"  ✗ Failed: {str(e)[:80]}...")
        
        if self.pipeline is None:
            raise RuntimeError("No model could be loaded. Try smaller size or check internet/storage.")
        
        self.is_llm = "LLM" in self.model_card
        print(f"Model type: {'LLM (supports lang)' if self.is_llm else 'CTC (zero-shot, no lang)'}")

    def download_youtube_audio(self, url):
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = self.audio_dir / f"audio_{timestamp}.wav"
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
            'outtmpl': str(output_path.with_suffix('')),
            'quiet': False,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            title = info.get('title', 'Unknown')
            duration = info.get('duration', 0)
        print(f"Downloaded: {title} ({duration}s)")
        return output_path, title, duration

    def split_audio_into_chunks(self, audio_path, max_sec=30):
        audio = AudioSegment.from_wav(audio_path)
        audio = audio.set_frame_rate(16000).set_channels(1)  # normalize
        duration_ms = len(audio)
        chunk_ms = max_sec * 1000
        chunks = []
        temp_files = []
        
        for i in range(0, duration_ms, chunk_ms):
            chunk = audio[i:i + chunk_ms]
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                chunk.export(tmp.name, format="wav")
                temp_files.append(tmp.name)
                chunks.append(tmp.name)
        print(f"Split into {len(chunks)} chunks")
        return chunks, temp_files

    def transcribe_chunk(self, chunk_path, lang="kik_Latn"):
        duration = len(AudioSegment.from_wav(chunk_path)) / 1000
        if duration > 40:
            return "[Chunk too long]"
        
        try:
            kwargs = {}
            if self.is_llm:
                kwargs["lang"] = [lang]
            trans = self.pipeline.transcribe([chunk_path], batch_size=1, **kwargs)
            return trans[0].strip() if trans else "[Empty]"
        except Exception as e:
            return f"[Error: {str(e)[:60]}]"

    def process_video(self, url, lang="kik_Latn"):
        audio_file, title, dur = self.download_youtube_audio(url)
        if not audio_file.exists():
            return None
        
        chunks, temps = self.split_audio_into_chunks(audio_file)
        transcriptions = []
        
        for chunk in tqdm(chunks, desc="Transcribing"):
            text = self.transcribe_chunk(chunk, lang)
            transcriptions.append(text)
        
        for t in temps:
            try: os.unlink(t)
            except: pass
        
        full_text = " ".join(transcriptions)
        
        result = {
            "title": title,
            "url": url,
            "duration": dur,
            "transcription": full_text,
            "model": self.model_card,
            "lang": lang if self.is_llm else "zero-shot (CTC)"
        }
        
        # Save
        safe_title = "".join(c if c.isalnum() else "_" for c in title)[:40]
        out_file = self.output_dir / f"{safe_title}_{datetime.now():%Y%m%d_%H%M}.json"
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        
        print(f"Saved: {out_file}")
        return result, full_text

# Gradio UI
def make_interface():
    transcriber = KikuyuTranscriber(model_size="300M", prefer_ctc=True)
    
    def transcribe(url, lang_code):
        try:
            result, text = transcriber.process_video(url, lang_code)
            preview = text[:600] + "..." if len(text) > 600 else text
            return (
                result["title"],
                f"{result['duration']} s",
                preview,
                text
            )
        except Exception as e:
            return "Error", str(e), "", ""
    
    with gr.Blocks() as demo:
        gr.Markdown("# Kikuyu YouTube Transcriber (Omnilingual ASR)")
        gr.Markdown(f"Model: **{transcriber.model_card}**")
        
        url = gr.Textbox(label="YouTube URL")
        lang = gr.Textbox(label="Lang code", value="kik_Latn")
        btn = gr.Button("Transcribe")
        
        title_out = gr.Textbox(label="Title")
        dur_out = gr.Textbox(label="Duration")
        preview_out = gr.Textbox(label="Preview", lines=6)
        full_out = gr.Textbox(label="Full Transcription", lines=12)
        
        btn.click(transcribe, [url, lang], [title_out, dur_out, preview_out, full_out])
    
    return demo

if __name__ == "__main__":
    demo = make_interface()
    demo.launch(share=True, debug=True, server_name="0.0.0.0")

KIKUYU YOUTUBE TRANSCRIBER - OMNILINGUAL ASR (FIXED)

🔧 Setting up environment...

✅ Packages installed!


2026-02-23 13:07:32.998681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771852053.022555    2271 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771852053.029851    2271 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771852053.048952    2271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771852053.048974    2271 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771852053.048977    2271 computation_placer.cc:177] computation placer alr


✅ Imports OK

GPU: Tesla P100-PCIE-16GB

Trying to load model (300M)...
  → omniASR_LLM_300M


Output()

✅ Loaded: omniASR_LLM_300M
Model type: LLM (supports lang)
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f42ebd620383acd821.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
